In [84]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [85]:
import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.append(str(root / "src"))

from notebook_cache import NotebookCache

# results and prints live on disk, so they survive kernel restarts; one folder per notebook
cache = NotebookCache(root / "cache" / "tracked")

In [86]:
import random

from gridworld import ACTIONS, STATES, TERMINALS, reset, sample_step, show

s = reset()
for _ in range(10):
    a = random.randrange(4)
    ns, r, done = sample_step(s, a)
    print(f"{s} --{ACTIONS[a]}-- {ns}  r={r}  done={done}")
    s = ns
    if done:
        break

(2, 0) -->-- (1, 0)  r=-0.04  done=False
(1, 0) --<-- (1, 0)  r=-0.04  done=False
(1, 0) --v-- (2, 0)  r=-0.04  done=False
(2, 0) --v-- (2, 0)  r=-0.04  done=False
(2, 0) -->-- (2, 1)  r=-0.04  done=False
(2, 1) --<-- (2, 1)  r=-0.04  done=False
(2, 1) --<-- (2, 0)  r=-0.04  done=False
(2, 0) -->-- (2, 1)  r=-0.04  done=False
(2, 1) -->-- (2, 2)  r=-0.04  done=False
(2, 2) --<-- (2, 1)  r=-0.04  done=False


In [ ]:
from common import greedy_policy, value_iteration

V = value_iteration()
pi = greedy_policy(V)

converged in 19 sweeps


In [88]:
s = reset()
for _ in range(10):
    a = pi[s]
    ns, r, done = sample_step(s, a)
    print(f"{s} --{ACTIONS[a]}-- {ns}  r={r}  done={done}")
    s = ns
    if done:
        break

(2, 0) --^-- (1, 0)  r=-0.04  done=False
(1, 0) --^-- (0, 0)  r=-0.04  done=False
(0, 0) -->-- (0, 1)  r=-0.04  done=False
(0, 1) -->-- (0, 2)  r=-0.04  done=False
(0, 2) -->-- (0, 3)  r=1.0  done=True


In [ ]:
def generate_episode(pi, max_steps=100):
    """[(state, action, reward), ...] — the agent's entire experience of one run."""
    episode = []
    s = reset(explore=True)
    for _ in range(max_steps):
        a = pi(s)
        ns, r, done = sample_step(s, a)
        episode.append((s, a, r))
        s = ns
        if done:
            break
    return episode

In [90]:
from collections import defaultdict


def mc_prediction(pi, n_episodes=5000, gamma=1.0):
    V_est = defaultdict(float)
    counts = defaultdict(int)
    for _ in range(n_episodes):
        episode = generate_episode(lambda s: pi[s])
        G = 0.0
        first_G = {}
        for s, a, r in reversed(episode):
            G = r + gamma * G
            first_G[s] = G
        for s, g in first_G.items():
            counts[s] += 1
            V_est[s] += (g - V_est[s]) / counts[s]
    return V_est, counts


V_mc, counts = mc_prediction(pi, n_episodes=5000)
show(V=V_mc)
show(V=V)
print({s: counts[s] for s in STATES if s not in TERMINALS})

+0.84  +0.90  +0.95  +1.00 
+0.79    ##   +0.65  -1.00 
+0.73  +0.68  +0.62  +0.38 

+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.75  +0.70  +0.65  +0.43 

{(0, 0): 3239, (0, 1): 3775, (0, 2): 4849, (1, 0): 2719, (1, 2): 1142, (2, 0): 2083, (2, 1): 1588, (2, 2): 1077, (2, 3): 561}
